[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Crew/transport-networks-lab/blob/main/Notebooks/Practica_guiada5.1_Isochrones.ipynb)


In [ ]:
%%capture
!pip install osmnx==1.9.4

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

import osmnx as ox
%matplotlib inline
ox.__version__
print(ox.__version__)
ox.settings.use_cache=True

# Descargar las calles de Buenos Aires

In [ ]:
# Definir el área de descarga
place_name = "Ciudad Autónoma de Buenos Aires, Argentina"

# Descargar la red vial (tipo 'drive' para calles transitables por vehículos)
G = ox.graph_from_place(place_name, network_type="drive")

In [ ]:
G.edges[list(G.edges)[0]]

In [ ]:
# Convertir a GeoDataFrame (nodos y aristas)
nodes, edges = ox.graph_to_gdfs(G)

print(f"Cantidad de calles: {len(edges)}")
edges.head()  # mostrar las primeras filas

In [ ]:
df_edges = edges.copy()
df_edges.reset_index(inplace=True)
df_edges = df_edges[['u', 'v', 'oneway', 'lanes', 'highway', 'maxspeed', 'length', 'geometry']]
df_edges.head()

In [ ]:
# Visualizar las calles
fig, ax = plt.subplots(figsize=(10, 10))
df_edges.plot(ax=ax, linewidth=0.5, color="black")
plt.title("Calles de la Ciudad Autónoma de Buenos Aires", fontsize=14)
plt.axis("off")
plt.show()

# Cálculo de tiempos de viaje

In [ ]:
# chequeo de datos nulos o igual a 0
print(f"Registros con distancias nulas: {len(df_edges.loc[df_edges['length'].isnull()])}")
print(f"Registros con distancias igual a 0: {len(df_edges.loc[df_edges['length'] == 0])}")
print("___________________")
print(f"Registros con velocidades nulas: {len(df_edges.loc[df_edges['maxspeed'].isnull()])}")
print(f"Registros con velocidades igual a 0: {len(df_edges.loc[df_edges['maxspeed'] == 0])}")

In [ ]:
# chequeo de tipo de datos
df_edges['maxspeed'].apply(lambda x: type(x).__name__).value_counts()

In [ ]:
# algunos valores
for t in df_edges['maxspeed'].apply(type).unique():
    print(f"\nEjemplos tipo {t}:")
    print(df_edges[df_edges['maxspeed'].apply(lambda x: isinstance(x, t))]['maxspeed'].head())

In [ ]:
# pasamos los valores de velocidad a númerido
df_edges['maxspeed_mod'] = df_edges.apply(lambda row: float(row.maxspeed[0]) if type(row.maxspeed) == list else float(row.maxspeed), axis=1)
df_edges['maxspeed_mod'].apply(lambda x: type(x).__name__).value_counts()

In [ ]:
# obtenemos la mediana de la velocidad para completar valores faltantes
vel_mediana = df_edges.loc[df_edges['maxspeed_mod'].notnull(), 'maxspeed_mod'].median()
print(f"Velocidad más común: {vel_mediana}")

In [ ]:
df_edges.loc[df_edges['maxspeed_mod'].isnull(), 'maxspeed_mod'] = vel_mediana

In [ ]:
# volvemos a chequear valores faltantes
print(f"Registros con velocidades nulas: {len(df_edges.loc[df_edges['maxspeed_mod'].isnull()])}")
print(f"Registros con velocidades igual a 0: {len(df_edges.loc[df_edges['maxspeed_mod'] == 0])}")

In [ ]:
# calculamos el tiempo de viaje a partir de la velocidad y distancia
df_edges['tiempo_min'] = round(((df_edges['length'] / 1000) / df_edges['maxspeed_mod']) * 60, 3)
df_edges.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

lw = np.clip(df_edges["tiempo_min"], 0.5, 10)  # grosor mínimo 0.5, máximo 10

df_edges.plot(
    ax=ax,
    column="tiempo_min",
    cmap="viridis",
    linewidth=lw,
    legend=True,
)

plt.title("Calles de Buenos Aires - Tiempo estimado de recorrido (minutos)", fontsize=13)
plt.axis("off")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
lw = np.clip(df_edges["maxspeed_mod"], 0.1, 1.5)

df_edges.plot(
    ax=ax,
    column="maxspeed_mod",
    cmap="viridis_r",
    linewidth=lw,
    legend=True,
)

plt.title("Calles de Buenos Aires - Velocidad máxima (km/h)", fontsize=13)
plt.axis("off")
plt.show()

# Reconstrucción del grafo

In [ ]:
df_edges.head()

## Opición 1: cargar tiempo de viaje en grafo existente

In [ ]:
# Reemplazar el atributo de peso en el grafo original
for u, v, k, data in G.edges(keys=True, data=True):
    data["tiempo_min"] = df_edges.loc[(df_edges.u == u) & (df_edges.v == v), "tiempo_min"].values[0]

In [ ]:
G.edges[list(G.edges)[0]]

## Opición 2: creamos el grafo a partir de dataframe (en este caso contando con los id de los nodos)

In [ ]:
# Aseguramos que el GeoDataFrame tenga coordenadas planas (en metros)
df_edges = df_edges.to_crs(3857)

In [ ]:
# Crear grafo dirigido a partir de DataFrame
H = nx.from_pandas_edgelist(
    df_edges,
    source="u",
    target="v",
    edge_attr=["length", "maxspeed", "tiempo_min", "oneway", "lanes", "highway"],
    create_using=nx.DiGraph()
)

In [ ]:
H.edges[list(H.edges)[0]]

## Opición 3: creamos el grafo a partir de dataframe (en este caso solo teniendo las geometrías)

In [ ]:
df_edges_2 = df_edges.copy()
df_edges_2 = df_edges_2[['oneway', 'lanes', 'highway', 'maxspeed', 'length', 'tiempo_min', 'geometry']]
df_edges_2.head()

In [ ]:
# Crear columnas de nodos (inicio y fin)
df_edges_2["u"] = df_edges_2.geometry.apply(lambda g: g.coords[0])
df_edges_2["v"] = df_edges_2.geometry.apply(lambda g: g.coords[-1])

# Crear grafo dirigido
I = nx.from_pandas_edgelist(
    df_edges_2,
    source="u",
    target="v",
    edge_attr=["length", "maxspeed", "tiempo_min", "oneway", "lanes", "highway"],
    create_using=nx.DiGraph()
)

In [ ]:
list(I.edges)[0]

In [ ]:
I.edges[list(I.edges)[0]]

# Generar Isocronas

## Determinar los nodos y ejes accesibles para los tiempos y origenes definidos

In [ ]:
# Selección de algunos nodos origenes aleatoriamente
origen = list(df_edges.sample(5)['u'])
print(f"Nodos origen: {origen}")

# Definición de un listado de tiempos límites
lista_limite_tiempo = [1, 3, 5]  # minutos
print(f"Límites de tiempo: {lista_limite_tiempo}")

In [ ]:
# Calcular la distancia mínima (peso acumulado) para toda la red
distancias, caminos = nx.multi_source_dijkstra(G, origen, weight="tiempo_min")

In [ ]:
list(distancias.items())[1000]

In [ ]:
# Asignar la distancia a cada nodo
nx.set_node_attributes(G, distancias, "tiempo_viaje")
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

In [ ]:
edges_gdf = edges_gdf.to_crs(3857)
nodes_gdf = nodes_gdf.to_crs(3857)

gdf_nodos_origen = nodes.reset_index()
gdf_nodos_origen = gdf_nodos_origen.loc[gdf_nodos_origen.osmid.isin(origen)]
gdf_nodos_origen_3857 = gdf_nodos_origen.to_crs(3857)

fig, ax = plt.subplots(figsize=(10, 10))


edges_gdf.plot(ax=ax, color="lightgray", linewidth=0.5)

nodes_gdf.plot(
    ax=ax,
    column="tiempo_viaje",
    cmap="viridis_r",
    markersize=30,
    legend=True,
    legend_kwds={"label": "Tiempo de viaje (minutos)"}
)
gdf_nodos_origen_3857.plot(ax=ax, color="red", markersize=60, label="Origen")

plt.title("Tiempo de viaje desde los orígenes (min)")
plt.axis("off")
plt.show()

## Generación de las isocronas para cada tiempo

In [ ]:
# Filtrar nodos dentro del límite
list_isocronas = []
for limite_tiempo in lista_limite_tiempo:
  nodos_isocrona = [n for n,d in distancias.items() if d <= limite_tiempo]

  # Subgrafo (solo la parte accesible dentro del límite)
  subG = G.subgraph(nodos_isocrona).copy()

  # Nodos y ejes de la subred
  nodes_subG, edges_subG = ox.graph_to_gdfs(subG)
  print(len(edges_subG))
  # Almacenamos los ejes para cada tiempo
  list_isocronas.append(edges_subG)


In [ ]:
list_isocronas[0]

In [ ]:
df_isocronas_1min = list_isocronas[0]
df_isocronas_3min = list_isocronas[1]
df_isocronas_5min = list_isocronas[2]

In [ ]:
# área alcanzada en el primer tiempo
df_isocronas_1min.to_crs(3857).buffer(100).unary_union

In [ ]:
# Generación de polígonos a partir de los ejes
list_isocronas_polys = []
for id in range(len(list_isocronas)):

  df = list_isocronas[id]
  df_geom = df.to_crs(3857).buffer(100).unary_union

  iso_union_gdf = gpd.GeoDataFrame(
    [{"geometry": df_geom, "tiempo": lista_limite_tiempo[id]}],
    crs="EPSG:3857"
  )

  list_isocronas_polys.append(iso_union_gdf)

In [ ]:
pd.concat(list_isocronas_polys)

In [ ]:
# Generación de anillos a partir de los polígonos
rings = []
prev_poly = None
for poly in list_isocronas_polys:
    if poly is None:
        rings.append(None)
        prev_poly = poly
        continue
    if prev_poly is None:
        rings.append(poly)
    else:
        rings.append(poly.difference(prev_poly))

In [ ]:
df_rings = pd.concat(rings)
df_rings

In [ ]:
# Ejes:
edges_3857 = df_edges.to_crs(3857)

# Nodos origen:
gdf_nodos_origen = nodes.reset_index()
gdf_nodos_origen = gdf_nodos_origen.loc[gdf_nodos_origen.osmid.isin(origen)]
gdf_nodos_origen_3857 = gdf_nodos_origen.to_crs(3857)

# Anillos:
ring_1 = df_rings.loc[df_rings.tiempo == 1]
ring_3 = df_rings.loc[df_rings.tiempo == 3]
ring_5 = df_rings.loc[df_rings.tiempo == 5]


In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))

edges_3857.plot(ax=ax, color="#adb5bd", linewidth=0.5, label="Calles")

ring_5.plot(ax=ax, color="#90e0ef", alpha=0.3, edgecolor="none", label="3–5 min")
ring_3.plot(ax=ax, color="#00b4d8", alpha=0.4, edgecolor="none", label="1–3 min")
ring_1.plot(ax=ax, color="#0077b6", alpha=0.5, edgecolor="none", label="0–1 min")

gdf_nodos_origen_3857.plot(ax=ax, color="#fca311", markersize=60, label="Origen")

plt.title("Isocronas (1, 3, 5 min) — Ciudad de Buenos Aires", fontsize=14)
plt.axis("off")
plt.legend(frameon=True, loc="lower right")

plt.show()